# Synthetic Patient Dataset Generator

This notebook generates a synthetic but realistic clinical trial dataset (`patients_raw.csv`).
Unlike simple random assignment, this generator uses statistical models (such as multinomial and logistic regression equations) to build **biologically plausible correlations** between demographic factors (Age, Sex) and clinical comorbidities (Smoking, Obesity, Diabetes, Cancer).

*Author: Marcos Caballero*  
*Target: Clinical Trial Randomization Engine (Phase 1)*


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# Set random seed for perfect reproducibility (as required for regulatory audits)
rng = np.random.default_rng(seed=42)
n_patients = 150

## 1. Age and Biological Sex

### Clinical & Statistical Context:
Clinical trials for chronic conditions (like type 2 diabetes or cardiovascular diseases) typically skew toward middle-aged and older populations. A flat uniform age distribution (e.g., equal numbers of 18-year-olds and 80-year-olds) does not reflect real-world clinical recruitment.
- **Biological Sex**: Standard 50/50 division is used as a baseline, representing typical randomized cohorts.

### Mathematical Formulation:
We model age using a **Beta distribution** ($	ext{Beta}(a=3, b=2)$), shifted and scaled to cover the adult age range of 18 to 85:
$$\text{Age} = 18 + 67 \times \text{Beta}(3, 2)$$

- **Rationale for Beta Parameters ($a=3, b=2$):** A Beta distribution with $a > b$ is left-skewed (meaning the mass is concentrated toward the upper end of the interval). 
- **Resulting Cohort Profile:** The theoretical mean is $18 + 67 \times \frac{3}{3+2} = 58.2$ years, with the majority of patients clustered in the 50–75 age range. This perfectly mimics the demographics of late-stage clinical trial participants.


In [ ]:
# Age (Beta distribution, skewed older)
ages = 18 + 67 * rng.beta(a=3, b=2, size=n_patients)
ages = np.round(ages, 1)

# Sex (Binomial 50/50)
sex_choices = ['Male', 'Female']
sex = rng.choice(sex_choices, size=n_patients, p=[0.5, 0.5])

## 2. Body Mass Index (BMI) and Obesity

### Clinical & Statistical Context:
According to the **CDC NHANES** data:
1. The age-adjusted prevalence of obesity (defined as $\text{BMI} \ge 30 \text{ kg/m}^2$) among U.S. adults is approximately **41.9%**.
2. Obesity rates are not flat across ages. NHANES data shows obesity peaks in middle age (40-59 years) at around **44.3%** and declines slightly in the geriatric cohort (60+ years) to **41.5%**, due to sarcopenia, cachexia, and cohort mortality effects.
3. BMI is notoriously **right-skewed** in the general population, with a long tail extending into severe obesity (Class III, $\text{BMI} \ge 40$).

### Modeling Decisions & Coefficients:
- **Baseline BMI**: We model base BMI using a log-normal distribution: $\text{BMI}_{\text{base}} = e^{\mathcal{N}(2.3, 0.09)} + 15$.
  - The parameter $loc=2.3$ yields a median exponential value of $e^{2.3} \approx 10$. Adding 15 sets the baseline median BMI around 25 (the boundary between normal weight and overweight). The standard deviation of $0.3$ on the log-scale creates a realistic right-skewed tail.
- **Age Effect**: We apply an age-dependent quadratic adjustment to represent the middle-age peak:
  $$\Delta\text{BMI}_{\text{age}} = 0.1 \times (\text{Age} - 18) - 0.0015 \times (\text{Age} - 18)^2$$
  - This quadratic curve peaks at $\text{Age} \approx 51$ (adding $+1.65 \text{ kg/m}^2$ of BMI) and falls back for younger and older ages, capturing the physiological rise and fall of weight across the lifespan.
- **Obesity**: Defined strictly as $\text{Obesity} = \text{Yes}$ if $\text{BMI} \ge 30$, matching WHO criteria.


In [ ]:
# Base log-normal BMI (Median ~25)
bmi_base_ln = rng.normal(loc=2.3, scale=0.3, size=n_patients)
bmi_base = np.exp(bmi_base_ln) + 15

# Quadratic age effect peaking around age 50
age_effect = 0.1 * (ages - 18) - 0.0015 * (ages - 18)**2
bmi = np.round(bmi_base + age_effect, 1)

# Ensure plausible biological limits (16.0 to 65.0)
bmi = np.clip(bmi, 16.0, 65.0)
obesity = np.where(bmi >= 30.0, 'Yes', 'No')

## 3. Smoking Status

### Clinical & Statistical Context:
According to the **CDC (2021)**:
1. Approximately **11.5%** of U.S. adults are current smokers.
2. Smoking is more common among men (**13.1%**) than women (**10.1%**).
3. The proportion of former smokers increases dramatically in older cohorts as individuals quit due to health concerns or mortality selection.

### Modeling Decisions & Coefficients:
We implement a **multinomial logistic model** (Softmax) to determine whether a patient is `Never`, `Former`, or `Current`.
We define logits (log-odds) for each state:
- $\text{logit}(\text{Never}) = 0.8$ (baseline reference).
- $\text{logit}(\text{Former}) = 0.2 + 0.02 \times (\text{Age} - 40) + \gamma_{\text{sex}}$
  - $\gamma_{\text{sex}} = +0.1$ for Males, $-0.1$ for Females (males have higher lifetime initiation rates).
  - The coefficient of $+0.02$ per year above age 40 models the cumulative probability of quitting as age increases.
- $\text{logit}(\text{Current}) = -0.2 - 0.03 \times (\text{Age} - 40) + \beta_{\text{sex}}$
  - $\beta_{\text{sex}} = +0.2$ for Males, $-0.2$ for Females (representing higher male active smoking rates).
  - The coefficient of $-0.03$ per year above age 40 captures the steady decline in active smoking among older populations.


In [ ]:
smoking = []
for i in range(n_patients):
    s = sex[i]
    a = ages[i]
    
    logit_never = 0.8
    logit_former = 0.2 + 0.02 * (a - 40) + (0.1 if s == 'Male' else -0.1)
    logit_current = -0.2 - 0.03 * (a - 40) + (0.2 if s == 'Male' else -0.2)
    
    logits = np.array([logit_never, logit_former, logit_current])
    probs = np.exp(logits) / np.sum(np.exp(logits))
    
    status = rng.choice(['Never', 'Former', 'Current'], p=probs)
    smoking.append(status)

smoking = np.array(smoking)

## 4. Diabetes (Type 2)

### Clinical & Statistical Context:
The **American Diabetes Association (ADA)** states:
1. About **11.3%** of the general U.S. population has diabetes.
2. Prevalence scales dramatically with age, reaching **29.2%** for adults aged 65+.
3. Obesity is the primary driver: individuals with obesity are **3 to 7 times** more likely to develop type 2 diabetes than those with normal BMI.

### Modeling Decisions & Coefficients:
We model the probability of diabetes using a logistic function:
$$P(\text{Diabetes} = \text{Yes}) = \frac{1}{1 + e^{-z}}$$
$$z = -4.0 + 0.04 \times (\text{Age} - 40) + 0.15 \times (\text{BMI} - 25)$$

- **Intercept ($-4.0$):** Sets a very low baseline probability ($P \approx 1.8\%$) for a young (40yo), normal-weight (BMI 25) individual.
- **Age Coefficient ($0.04$):** Models a $4\%$ increase in log-odds per year, capturing the age-dependent decline in insulin sensitivity.
- **BMI Coefficient ($0.15$):** For every 1-unit increase in BMI above normal, log-odds increase by $0.15$. 
  - For a patient with Class I Obesity (BMI 35), their odds of diabetes increase by $e^{0.15 \times 10} = e^{1.5} \approx 4.48$ times compared to a normal weight peer. This matches the **3-to-7-fold relative risk** reported in epidemiological studies.


In [ ]:
z_diabetes = -4.0 + 0.04 * (ages - 40) + 0.15 * (bmi - 25)
p_diabetes = 1 / (1 + np.exp(-z_diabetes))
diabetes = rng.binomial(1, p_diabetes)
diabetes = np.where(diabetes == 1, 'Yes', 'No')

## 5. Cancer and Tumor Stage

### Clinical & Statistical Context:
According to the **National Cancer Institute (NCI)**:
1. Age is the single most important risk factor for cancer overall. The median age of diagnosis is **66 years**.
2. Tobacco use is the leading cause of cancer and cancer death, accounting for about **85% of lung cancers** and substantially elevating overall cancer rates.
3. Obesity is linked to higher risk of 13 types of cancer, resulting in a **1.5 to 2.0-fold** relative risk.

### Modeling Decisions & Coefficients:
We model cancer probability via:
$$P(\text{Cancer} = \text{Yes}) = \frac{1}{1 + e^{-w}}$$
$$w = -3.5 + 0.06 \times (\text{Age} - 45) + 1.2 \times \mathbb{I}(\text{Current}) + 0.6 \times \mathbb{I}(\text{Former}) + 0.5 \times \mathbb{I}(\text{Obesity})$$

- **Intercept ($-3.5$):** Reflects a baseline probability of $2.9\%$ for a 45-year-old non-smoker of normal weight.
- **Age Coefficient ($0.06$):** Increases the log-odds of cancer by $0.6$ per decade (approx. $1.8$-fold increase in odds per decade), simulating the lifetime accumulation of somatic mutations.
- **Smoking Coefficients ($1.2$ for Current, $0.6$ for Former):** A current smoker experiences a $e^{1.2} \approx 3.3$-fold increase in cancer odds, while a former smoker retains a $e^{0.6} \approx 1.8$-fold elevated odds, illustrating smoking-related damage and partial risk recovery.
- **Obesity Coefficient ($0.5$):** Translates to a $e^{0.5} \approx 1.65$-fold increase in cancer odds, fitting the $1.5\text{-}2.0$ relative risk seen clinically.
- **Cancer Stage**: If a patient is diagnosed with cancer, they are assigned a stage from a multinomial distribution biased toward earlier stages ($I=35\%$, $II=30\%$, $III=20\%$, $IV=15\%$).


In [ ]:
is_current = (smoking == 'Current').astype(int)
is_former = (smoking == 'Former').astype(int)
is_obese = (obesity == 'Yes').astype(int)

# Logistic model for cancer probability
z_cancer = -3.5 + 0.06 * (ages - 45) + 1.2 * is_current + 0.6 * is_former + 0.5 * is_obese
p_cancer = 1 / (1 + np.exp(-z_cancer))
cancer_bin = rng.binomial(1, p_cancer)
cancer = np.where(cancer_bin == 1, 'Yes', 'No')

cancer_stage = []
stage_choices = ['I', 'II', 'III', 'IV']
stage_probs = [0.35, 0.30, 0.20, 0.15]
for c in cancer_bin:
    if c == 1:
        cancer_stage.append(rng.choice(stage_choices, p=stage_probs))
    else:
        cancer_stage.append('None')

## 6. Assembly and Export

We compile these interdependent variables into a structured pandas DataFrame. The output is written to `patients_raw.csv` at the root directory.


In [ ]:
patient_ids = [f"PAC-{i:03d}" for i in range(1, n_patients + 1)]

df = pd.DataFrame({
    'Patient_ID': patient_ids,
    'Age': ages,
    'Sex': sex,
    'BMI': bmi,
    'Obesity': obesity,
    'Smoking': smoking,
    'Diabetes': diabetes,
    'Cancer': cancer,
    'Cancer_Stage': cancer_stage
})

output_path = "patients_raw.csv"
df.to_csv(output_path, index=False)
print(f"Dataset assembled successfully. Shape: {df.shape}")
print(df.head())

## 7. Statistical Audit of Generated Correlations

To audit our mathematical models, we print the empirical correlations observed in our generated dataset. If the code was successful, we expect to see:
1. Diabetics showing a higher average BMI than non-diabetics.
2. Cancer patients showing an older average age.
3. Smoker status correlating with a higher prevalence of cancer.


In [ ]:
print(f"Mean BMI for Diabetics vs Non-Diabetics:")
print(df.groupby('Diabetes')['BMI'].mean())
print("\n" + "-"*40 + "\n")

print(f"Mean Age for Cancer vs Non-Cancer Patients:")
print(df.groupby('Cancer')['Age'].mean())
print("\n" + "-"*40 + "\n")

print(f"Cancer Prevalence by Smoking Status:")
print(pd.crosstab(df['Smoking'], df['Cancer'], normalize='index'))